In [3]:
import requests

# 1. Resolve the inference service URL
result = !kubectl get inferenceservices gpt-oss-20b-vllm \
    -o jsonpath='{.status.address.url}'

HOSTED_VLLM_API_BASE = result[0].strip()
print(f"Service URL: {HOSTED_VLLM_API_BASE}")
# e.g. http://gpt-oss-20b-vllm.kubeflow-${USERNAME}.svc.cluster.local



http://gpt-oss-20b-vllm.kubeflow-m-xochicale.svc.cluster.local


In [4]:
import requests

api_endpoint = HOSTED_VLLM_API_BASE+"/v1/chat/completions"

payload = {
    "model": "/mnt/models/models/gpt-oss-20b",
    "messages": [
        {"role": "user", "content": "Translate to Portugeues, German and Spanish: How old are you?,"}
    ],
    "max_tokens": 500
}

response = requests.post(api_endpoint, json=payload)
print(response.json())

{'id': 'chatcmpl-0c91ef9b-3333-4b0b-953a-986c2a354c8a', 'object': 'chat.completion', 'created': 1780492664, 'model': '/mnt/models/models/gpt-oss-20b', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': '**Portuguese (informal):**  \n*Quantos anos você tem?*  \n\n**German (informal):**  \n*Wie alt bist du?*  \n\n**Spanish (informal):**  \n*¿Cuántos años tienes?*', 'refusal': None, 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [], 'reasoning': 'We need to translate "How old are you?" into Portuguese, German, and Spanish. Provide translations and perhaps note form? The user just says translate to Portuguese, German and Spanish: How old are you? So answer likely: Portuguese: "Quantos anos você tem?" or "Quantos anos tem?" German: "Wie alt bist du?" Spanish: "¿Cuántos años tienes?" We can give all three. Maybe mention being formal or informal. But likely just translations. So answer: Portuguese: "Quantos anos você tem?" German: "Wie alt bist du?"

In [5]:
# 2. Build and send the chat-completion request 
API_ENDPOINT = f"{HOSTED_VLLM_API_BASE}/v1/chat/completions"
MODEL_PATH    = "/mnt/models/models/gpt-oss-20b"

payload = {
    "model": MODEL_PATH,
    "messages": [
        {
            "role": "user",
            "content": (
                "Translate the following sentence into Portuguese, German, and Spanish.\n"
                "Return each translation on a separate line, labelled by language.\n\n"
                "Sentence: How old are you?"
            ),
        }
    ],
    "max_tokens": 500,
    "temperature": 0.2,   # low temp → consistent, literal translations
}

response = requests.post(API_ENDPOINT, json=payload, timeout=60)
response.raise_for_status()           # surface HTTP errors immediately


Portuguese: Quantos anos você tem?  
German: Wie alt bist du?  
Spanish: ¿Cuántos años tienes?


In [ ]:
# 3. Display the result
data    = response.json()
message = data["choices"][0]["message"]["content"]
print(message)